In [ ]:
!pip install -U langchain-ollama langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 24.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.4 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/review_dataset

review_dataset	w_review_train.parquet


In [ ]:
import duckdb

con = duckdb.connect(database=':memory:', read_only=False)

In [ ]:
df = con.execute("SELECT * FROM read_csv_auto('/content/drive/MyDrive/review_dataset/review_dataset/w_review_train.csv', parallel=false, encoding='UTF-8', ignore_errors=true)").fetchdf()

df.head()

,column0,column1
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,3
1,อาหารที่นี่เป็นอาหารจีนแคะที่หากินยากในบ้านเรา...,4
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,3
3,รัานคัพเค้กในเมืองไทยมีไม่มาก หลายๆคนอาจจะสงสั...,5
4,อร่อย!!! เดินผ่านDigital gatewayทุกวัน ไม่ยักร...,5


In [ ]:
df.shape

(40000, 2)

In [ ]:
con.execute("COPY df TO '/content/drive/MyDrive/review_dataset/w_review_train.parquet' (FORMAT PARQUET)")

In [ ]:
parquet_avg_rating_direct = con.execute("SELECT AVG(column1) FROM '/content/drive/MyDrive/review_dataset/w_review_train.parquet' WHERE column0 LIKE '%อร่อย%'").fetchone()[0]
parquet_avg_rating_direct

3.8313027179006562

In [ ]:
parquet_file_path = '/content/drive/MyDrive/review_dataset/w_review_train.parquet'

In [ ]:
parquet_avg_rating_direct = con.execute(f"SELECT AVG(column1) FROM '{parquet_file_path}' WHERE column0 LIKE '%อร่อย%'").fetchone()[0]
parquet_avg_rating_direct

3.8313027179006562

In [ ]:
coffee_keywords = [
    'กาแฟ', 'คาเฟ่', 'coffee', 'cafe', 'เอสเปรสโซ่', 'ลาเต้',
    'มอคค่า', 'คาปูชิโน่', 'เฟรนช์เพรส', 'cold brew', 'drip',
    'espresso', 'latte', 'americano', 'macchiato', 'brew',
    'specialty', 'slow bar', 'ร้านกาแฟ', 'ร้านคาเฟ่', 'กาแฟสด',
    'เมนูกาแฟ', 'กาแฟนม', 'กาแฟดำ', 'ร้านบาริสต้า', 'barista', 'coffee shop'
]

where_clause = " OR ".join([f"column0 LIKE '%{keyword}%'" for keyword in coffee_keywords])

In [ ]:
%%time
coffee_reviews_df = con.execute(f"SELECT * FROM '{parquet_file_path}' WHERE {where_clause}").fetchdf()

CPU times: user 1.85 s, sys: 49.6 ms, total: 1.9 s
Wall time: 1.91 s


In [ ]:
csv_file_path = '/content/drive/MyDrive/review_dataset/w_review_train.csv'

In [ ]:
%%time
coffee_reviews_df_csv = con.execute(f"SELECT * FROM read_csv_auto('/content/drive/MyDrive/review_dataset/review_dataset/w_review_train.csv', parallel=false, encoding='UTF-8', ignore_errors=true) WHERE {where_clause}").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 2.27 s, sys: 56.2 ms, total: 2.32 s
Wall time: 2.44 s


In [ ]:
coffee_reviews_df.head()

,column0,column1
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,3
1,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...,4
2,สารภาพว่าไม่เคยคิดจะไปต่อคิวซื้อมากินเองครับ บ...,3
3,ร้านอาหารญี่ปุ่นร้านนี้ ใจจริงไม่อยากแนะนำเลยค...,5
4,เดือนแรกที่เค้าต่อคิวกัน 2 - 3 ชั่วโมง เราก็ว่...,5


In [ ]:
coffee_reviews_df.shape

(4981, 2)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [ ]:
!ollama pull gemma3:270m

In [ ]:
!ollama list

NAME           ID              SIZE      MODIFIED      
gemma3:270m    e7d36fb2c3b3    291 MB    5 seconds ago    


In [ ]:
!ollama run gemma3:270m "สวัสดี"

สวัสดีค่ะ 😊



In [ ]:
!curl -s http://127.0.0.1:11434/api/tags

{"models":[{"name":"gemma3:270m","model":"gemma3:270m","modified_at":"2025-11-14T16:30:24.720589454Z","size":291554930,"digest":"e7d36fb2c3b3293cfe56d55889867a064b3a2b22e98335f2e6e8a387e081d6be","details":{"parent_model":"","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"268.10M","quantization_level":"Q8_0"}}]}

In [ ]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
llm = OllamaLLM(model="gemma3:270m")

In [ ]:
response = llm.invoke("สวัสดี")
print(response)

สวัสดีค่ะ 😊


In [ ]:
template = """
คุณเป็นครูสอน Python
คำถาม : {question}
"""

In [ ]:
prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)

In [ ]:
formatted_prompt = prompt.format(question="เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10")
print(formatted_prompt)


คุณเป็นครูสอน Python
คำถาม : เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10



In [ ]:
chain = prompt | llm

In [ ]:
response = chain.invoke({"question": "เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10"})

In [ ]:
response

'```python\ndef find_sum_of_numbers(numbers):\n  """\n  หาผลรวมของตัวเลข 1 ถึง 10\n  """\n  sum = 0\n  for number in numbers:\n    sum += number\n  return sum\n\n# ตัวอย่างการใช้งาน\nnumbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]\nresult = find_sum_of_numbers(numbers)\nprint("ผลรวมของตัวเลข:", result)\n```\n\n**คำอธิบาย:**\n\n1.  **`def find_sum_of_numbers(numbers):`**:  กำหนดชื่อ funzione `find_sum_of_numbers` ซึ่งจะถูกเรียกใช้เมื่อมีการเรียกใช้โค้ด\n2.  **`sum = 0`**:  สร้างค่า `sum` เป็น 0 เพื่อเก็บผลรวมของตัวเลขที่ได้\n3.  **`for number in numbers:`**:  วนซ้ำผ่านแต่ละตัวเลขใน `numbers`  (ซึ่งเป็น iterable ของตัวเลข)\n4.  **`sum += number`**:  เมื่อ `number` เป็นตัวเลขแล้ว  `sum` จะเพิ่มค่าเดิม `number` เข้าไปใน `sum`\n5.  **`return sum`**:  เมื่อ `sum` เป็นค่าคงที่แล้ว  ผลรวมของตัวเลขทั้งหมดคือ `sum`\n6.  **`numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`**:  สร้าง iterable ของตัวเลขทั้งหมดใน `numbers`\n7.  **`result = find_sum_of_numbers(numbers)`**:  เรียกใช้ฟังก์ชัน `find_sum_of_numbers` 

In [ ]:
from IPython.display import display, Markdown

In [ ]:
display(Markdown(response))

```python
def find_sum_of_numbers(numbers):
  """
  หาผลรวมของตัวเลข 1 ถึง 10
  """
  sum = 0
  for number in numbers:
    sum += number
  return sum

# ตัวอย่างการใช้งาน
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
result = find_sum_of_numbers(numbers)
print("ผลรวมของตัวเลข:", result)
```

**คำอธิบาย:**

1.  **`def find_sum_of_numbers(numbers):`**:  กำหนดชื่อ funzione `find_sum_of_numbers` ซึ่งจะถูกเรียกใช้เมื่อมีการเรียกใช้โค้ด
2.  **`sum = 0`**:  สร้างค่า `sum` เป็น 0 เพื่อเก็บผลรวมของตัวเลขที่ได้
3.  **`for number in numbers:`**:  วนซ้ำผ่านแต่ละตัวเลขใน `numbers`  (ซึ่งเป็น iterable ของตัวเลข)
4.  **`sum += number`**:  เมื่อ `number` เป็นตัวเลขแล้ว  `sum` จะเพิ่มค่าเดิม `number` เข้าไปใน `sum`
5.  **`return sum`**:  เมื่อ `sum` เป็นค่าคงที่แล้ว  ผลรวมของตัวเลขทั้งหมดคือ `sum`
6.  **`numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`**:  สร้าง iterable ของตัวเลขทั้งหมดใน `numbers`
7.  **`result = find_sum_of_numbers(numbers)`**:  เรียกใช้ฟังก์ชัน `find_sum_of_numbers` เพื่อหาผลรวมของตัวเลข
8.  **`print("ผลรวมของตัวเลข:", result)`**:  แสดงผลลัพธ์ที่ได้จากฟังก์ชัน
9.  **`# ตัวอย่างการใช้งาน`**:  มีการนำโค้ดไปใช้ในตัวอย่างการใช้งาน

**ข้อดีของโค้ดนี้:**

*   **ง่ายต่อเข้าใจ:**  โค้ดนี้มีโครงสร้างที่ชัดเจนและเข้าใจง่าย ทำให้ง่ายต่อการเรียนรู้
*   **ทำงานได้:**  โค้ดนี้ทำงานได้อย่างถูกต้องและมีประสิทธิภาพ
*   **มีประสิทธิภาพ:**  โค้ดนี้ทำงานได้เร็วกว่าวิธีอื่นๆ ในการหาผลรวมของตัวเลข

**ข้อควรระวัง:**

*   **ประสิทธิภาพ:**  การหาผลรวมของตัวเลขใน Python อาจจะซับซ้อนกว่าวิธีอื่นๆ ในบางกรณี  หากคุณต้องการประสิทธิภาพสูงมาก อาจต้องใช้ฟังก์ชันอื่นๆ ที่มีประสิทธิภาพกว่า
*   **การจัดการตัวแปร:**  การใช้ตัวแปรในโค้ดนี้อาจต้องมีการจัดการตัวแปรเพิ่มเติมเพื่อให้ได้ผลลัพธ์ที่ถูกต้อง
*   **ข้อจำกัด:**  การหาผลรวมของตัวเลขใน Python ยังคงเป็นเรื่องที่ซับซ้อน และอาจต้องใช้การปรับแก้บางอย่างเพื่อให้ได้ผลลัพธ์ที่ตรงกับความต้องการ



In [ ]:
prompt1 = PromptTemplate(
    template="แปลโจทย์เกี่ยวกับ {question}"
)

chain1 = prompt1 | llm

In [ ]:
prompt2 = PromptTemplate(
    template="ทำความเข้าใจโจทย์ที่แปลแล้วต่อไปนี้ {translate}"
)

chain2 = prompt2 | llm

In [ ]:
prompt3 = PromptTemplate(
    template="แก้โจทย์ตามที่เข้าใจต่อไปนี้ {understand}"
)

chain3 = prompt3 | llm

In [ ]:
prompt4 = PromptTemplate(
    template="อธิบายวิธิีทำจากผลลัทธ์ต่อไปนี้ {solve}"
)

chain4 = prompt4 | llm

In [ ]:
parser_chain = StrOutputParser()

In [ ]:
full_chain = chain1 | chain2 | chain3 | chain4 | parser_chain

In [ ]:
result = full_chain.invoke({"question":"2x + 5 = 15"})

In [ ]:
result

'วิธิีทำจากผลลัทธ์ มีลักษณะสำคัญดังนี้:\n\n*   **ความรู้และความเข้าใจ:** วิธิีมุ่งมั่นที่จะเข้าใจความรู้และเข้าใจแนวคิดต่างๆ ที่เกี่ยวข้องกับผลลัทธ์ รวมถึงผลกระทบของการเปลี่ยนแปลงผลลัทธิจารณ์\n*   **การวิเคราะห์:** วิธิีใช้ผลลัทธ์เพื่อวิเคราะห์สถานการณ์และผลลัพธ์ต่างๆ ของผลลัทธิจารณ์\n*   **การสร้างความเข้าใจ:** วิธิีใช้ผลลัทธ์เพื่อสร้างความเข้าใจเกี่ยวกับผลลัทธิจารณ์และแนวคิดที่เกี่ยวข้อง\n*   **การแก้ปัญหา:** วิธิีใช้ผลลัทธ์เพื่อแก้ปัญหาที่เกี่ยวข้องกับผลลัทธิจารณ์\n\nโดยสรุปแล้ว วิธิีทำจากผลลัทธ์เพื่อพัฒนาความรู้ ความเข้าใจ และการแก้ปัญหาเกี่ยวกับผลลัทธิจารณ์'

In [ ]:
prompt_template = """
[1. Role/Context บอก LLM ว่าควรทำตัวเป็นอะไร]
คุณเป็น AI ที่เชี่ยวชาญในการวิเคราะห์รีวิวภาษาไทย

[2. Task Description บอกงานที่ต้องทำ]
จาก Review ต่อไปนี้ ให้ Extract 3 keywords
ที่เป็นตัวแทนของ Review นี้ได้ดีที่สุด

[3. Input Data ข้อมูลที่ต้องประมวลผล]
Review: {review}

[4. Output Format บอกรูปแบบของคำตอบ]
ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'keywords'
เช่น {{"keywords": "กาแฟ, บรรยากาศ, ราคา"}

[5. Constraints ข้อจำกัดหรือเงื่อนไข]
- ต้องเป็นคำภาษาไทย
- คั่นด้วย comma
- หากไม่พบให้ตอบ 'ไม่ระบุ'

[6. Examples (optional) ตัวอย่าง input/output]
ตัวอย่าง
Input: "กาแฟอร่อย บรรยากาศดี ราคาแพง"
Output: {"keywords": "กาแฟ, บรรยากาศ, ราคา"}
"""

In [ ]:
prompt = """
Extract keywords from this review
{review}
"""

# Pros Simple, fast
# Cons อาจได้ผลไม่ดีกับ Tasks ซับซ้อน

In [ ]:
prompt = """
Extract keywords from reviews.

Example 1
Review: "กาแฟอร่อย บริการดี"
Keywords: กาแฟ, บริการ

Example 2
Review: "ราคาแพง แต่คุ้มค่า"
Keywords: ราคา, คุณภาพ

Now extract from this review
Review: {review}
Keywords:
"""

# Pros ได้ผลดีกว่า Zero-shot
# Cons Prompt ใช้ Tokens มากขึ้น

In [ ]:
prompt = """
Review: {review}

ขั้นตอนการ Extract keywords
1. อ่านรีวิวและเข้าใจความหมาย
2. ระบุหัวข้อหลักที่กล่าวถึง
3. เลือก 3 keywords ที่สำคัญที่สุด
4. ตอบในรูปแบบ JSON: {{"keywords": "คำ1, คำ2, คำ3"}}

ให้ทำทีละขั้นตอน
"""

# Pros ได้ผลดีกับ Reasoning tasks
# Cons ใช้ Tokens มากกว่า ช้ากว่า

In [ ]:
keyword_prompt = PromptTemplate(
template = """
จาก Review ต่อไปนี้
Extract 3 keywords ที่เป็นตัวแทนของ Review นี้

Review: {review}

ขั้นตอนการ Extract keywords
1. อ่านรีวิวและเข้าใจความหมาย
2. ระบุหัวข้อหลักที่กล่าวถึง
3. เลือก 3 keywords ที่สำคัญที่สุด
4. ตอบในรูปแบบ JSON: {{"keywords": "คำ1, คำ2, คำ3"}}

ให้ทำทีละขั้นตอน
""")

In [ ]:
sample_reviews = coffee_reviews_df['column0'].head(2)
sample_reviews

,column0
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...
1,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...


In [ ]:
sample_reviews.to_csv('/content/sample_reviews.csv', index=False)

In [ ]:
keyword_extract = keyword_prompt | llm | StrOutputParser()

In [ ]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

CPU times: user 30.1 ms, sys: 5.54 ms, total: 35.6 ms
Wall time: 3.07 s


In [ ]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text ```json
{"keywords": ["ร้านอาหาร", "ห้องน้ำ", "กาแฟ", "น้ำผึ้ง", "น้ำผึ้ง", "ลาบไข่ต้ม", "ไข่มันคาว", "ขนมไทย", "ขนมหวาน", "ฟรี", "ราคา", "เกินไป", "รับไม่ไหว", "ขนมไทย", "ไข่อร่อย", "เห้ยแต่ราคา", "เลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย", "ขนมไทย", "ไข่มันคาว", "ขนมไทย"]}
```

Review วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์

ก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก ...
text ```json
{
  "keywords": [
    "ลาเต้",
    "ร้อน",
    "กาแฟ"
  ]
}
```



In [ ]:
!ollama pull scb10x/typhoon2.1-gemma3-4b

In [ ]:
!ollama list

NAME                                  ID              SIZE      MODIFIED      
scb10x/typhoon2.1-gemma3-4b:latest    8cfab9097c9d    2.6 GB    7 seconds ago    
gemma3:270m                           e7d36fb2c3b3    291 MB    4 minutes ago    


In [ ]:
llm3 = OllamaLLM(model="scb10x/typhoon2.1-gemma3-4b")

In [ ]:
keyword_extract = keyword_prompt | llm3 | StrOutputParser()

In [ ]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

CPU times: user 102 ms, sys: 10.9 ms, total: 113 ms
Wall time: 17 s


In [ ]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text แน่นอนครับ ผมจะทำตามขั้นตอนที่คุณให้มาเพื่อดึงคำสำคัญจากรีวิวนี้ครับ

ขั้นตอนที่ 1: อ่านรีวิวและเข้าใจความหมาย

รีวิวนี้พูดถึงร้านอาหารที่มีขนาดใหญ่มาก, การตกแต่งที่ดูแปลก, อาหารบางอย่างอร่อย (ทอดมันหัวปลี, พะแนงห่อไข่) แต่มีราคาที่สูงเกินไป (โดยเฉพาะพะแนงห่อไข่), และมีของหวานฟรีในช่วงท้าย

ขั้นตอนที่ 2: ระบุหัวข้อหลักที่กล่าวถึง

หัวข้อหลักที่กล่าวถึงในรีวิวนี้ ได้แก่:
*   ขนาดร้าน
*   ราคาอาหาร
*   รสชาติอาหาร (โดยเฉพาะทอดมันหัวปลีและพะแนงห่อไข่)
*   ของหวานฟรี

ขั้นตอนที่ 3: เลือก 3 keywords ที่สำคัญที่สุด

จากหัวข้อหลักที่กล่าวถึง ผมจะเลือกคำสำคัญ 3 คำ ได้แก่:

*   ขนาดใหญ่
*   ราคาแพง
*   รสชาติ

ขั้นตอนที่ 4: ตอบในรูปแบบ JSON

```json
{
  "keywords": ["ขนาดใหญ่", "ราคาแพง", "รสชาติ"]
}
```

หวังว่าคำตอบนี้จะเป็นประโยชน์นะครับ หากคุณมีคำถามเพิ่มเติมหรือต้องการให้ผมช่วยอะไรอีก โปรดแจ้งให้ทราบได้เลยครับ

Review วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์

ก็ได้ย

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class ReviewKeywords(BaseModel):
    keywords: str = Field(
        ...,  # Required field
        description="3 keywords separated by comma",
        examples=["กาแฟ, บรรยากาศ, ราคา"]
    )

class KeywordsExtraction(BaseModel):
    extracted_qualities: ReviewKeywords

In [ ]:
keywords_parser = JsonOutputParser(pydantic_object=KeywordsExtraction)

In [ ]:
keyword_extract = keyword_prompt | llm3 | keywords_parser

In [ ]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    results.append({"review": review, "text": text})

CPU times: user 146 ms, sys: 15 ms, total: 161 ms
Wall time: 13.7 s


In [ ]:
results[0]['text']['keywords']

['ราคา', 'รสชาติ', 'ขนาด']

In [ ]:
results[1]['text']['keywords']

['กาแฟ', 'บรรยากาศ', 'วาวี']

In [ ]:
llm3 = OllamaLLM(
    model="scb10x/typhoon2.1-gemma3-4b",
    temperature=0.05
)

keyword_extract = keyword_prompt | llm3 | keywords_parser

In [ ]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    results.append({"review": review, "text": text})

CPU times: user 138 ms, sys: 8.97 ms, total: 147 ms
Wall time: 13.2 s


In [ ]:
results[0]['text']['keywords'], results[1]['text']['keywords']

(['ราคา', 'อาหาร', 'ขนาด'], ['กาแฟ', 'บรรยากาศ', 'รสชาติ'])

In [ ]:
keyword_prompt = PromptTemplate(
    template="""จาก Review ต่อไปนี้
    Extract keywords 3 keywords เท่านั้น ที่เป็นตัวแทนของ Review นี้ได้ดีที่สุด

    Review: {review}

    ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'keywords'
    และค่าเป็นข้อความที่ดึงได้คั่นด้วย comma
    เช่น {{"keywords": "กาแฟ, บรรยากาศ, ราคา"}}

    หากไม่พบข้อมูลให้ตอบ 'ไม่ระบุ'
    """
)

In [ ]:
keyword_extract = keyword_prompt | llm3 | keywords_parser

In [ ]:
inputs = [{"review": r} for r in sample_reviews]

In [ ]:
%%time
results = keyword_extract.batch(inputs)

CPU times: user 20.2 ms, sys: 1.15 ms, total: 21.4 ms
Wall time: 3.07 s


In [ ]:
results

[{'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
 {'keywords': 'กาแฟ, บรรยากาศ, รสชาติ'}]

In [ ]:
sample_reviews_100 = coffee_reviews_df['column0'].iloc[0:200].tolist()
inputs_100 = [{"review": r} for r in sample_reviews_100]

BATCH_SIZE = 8

results_100 = []

In [ ]:
from tqdm import tqdm

In [ ]:
for i in tqdm(range(0, len(inputs_100), BATCH_SIZE), desc="Extracting keywords"):
    batch = inputs_100[i:i+BATCH_SIZE]
    outputs = keyword_extract.batch(batch)
    for inp, out in zip(batch, outputs):
        results_100.append({"review": inp["review"], "text": out})


Extracting keywords: 100%|██████████| 25/25 [04:07<00:00,  9.91s/it]


In [ ]:
import pandas as pd
df = pd.DataFrame(results_100)
df.to_csv("results_part1.csv", index=False, encoding='utf-8-sig')

In [ ]:
sample_reviews_100 = coffee_reviews_df['column0'].iloc[200:400].tolist()
inputs_100 = [{"review": r} for r in sample_reviews_100]
BATCH_SIZE = 8
results_100 = []

In [ ]:
for i in tqdm(range(0, len(inputs_100), BATCH_SIZE), desc="Extracting keywords"):
    batch = inputs_100[i:i+BATCH_SIZE]
    outputs = keyword_extract.batch(batch)
    for inp, out in zip(batch, outputs):
        results_100.append({"review": inp["review"], "text": out})


Extracting keywords: 100%|██████████| 25/25 [03:59<00:00,  9.57s/it]


In [ ]:
import pandas as pd
df = pd.DataFrame(results_100)
df.to_csv("results_part2.csv", index=False, encoding='utf-8-sig')

In [ ]:
sample_reviews_100 = coffee_reviews_df['column0'].iloc[400:600].tolist()
inputs_100 = [{"review": r} for r in sample_reviews_100]
BATCH_SIZE = 8
results_100 = []

In [ ]:
for i in tqdm(range(0, len(inputs_100), BATCH_SIZE), desc="Extracting keywords"):
    batch = inputs_100[i:i+BATCH_SIZE]
    outputs = keyword_extract.batch(batch)
    for inp, out in zip(batch, outputs):
        results_100.append({"review": inp["review"], "text": out})


Extracting keywords: 100%|██████████| 25/25 [03:53<00:00,  9.34s/it]


In [ ]:
import pandas as pd
df = pd.DataFrame(results_100)
df.to_csv("results_part3.csv", index=False, encoding='utf-8-sig')

In [ ]:
sample_reviews_100 = coffee_reviews_df['column0'].iloc[600:800].tolist()
inputs_100 = [{"review": r} for r in sample_reviews_100]
BATCH_SIZE = 8
results_100 = []

In [ ]:
for i in tqdm(range(0, len(inputs_100), BATCH_SIZE), desc="Extracting keywords"):
    batch = inputs_100[i:i+BATCH_SIZE]
    outputs = keyword_extract.batch(batch)
    for inp, out in zip(batch, outputs):
        results_100.append({"review": inp["review"], "text": out})


Extracting keywords: 100%|██████████| 25/25 [03:56<00:00,  9.46s/it]


In [ ]:
import pandas as pd
df = pd.DataFrame(results_100)
df.to_csv("results_part4.csv", index=False, encoding='utf-8-sig')

In [ ]:
sample_reviews_100 = coffee_reviews_df['column0'].iloc[800:1000].tolist()
inputs_100 = [{"review": r} for r in sample_reviews_100]
BATCH_SIZE = 8
results_100 = []

In [ ]:
for i in tqdm(range(0, len(inputs_100), BATCH_SIZE), desc="Extracting keywords"):
    batch = inputs_100[i:i+BATCH_SIZE]
    outputs = keyword_extract.batch(batch)
    for inp, out in zip(batch, outputs):
        results_100.append({"review": inp["review"], "text": out})


Extracting keywords: 100%|██████████| 25/25 [03:49<00:00,  9.16s/it]


In [ ]:
import pandas as pd
df = pd.DataFrame(results_100)
df.to_csv("results_part5.csv", index=False, encoding='utf-8-sig')

In [ ]:
from tqdm import tqdm

In [ ]:
for i in tqdm(range(0, len(inputs_100), BATCH_SIZE), desc="Extracting keywords"):
    batch = inputs_100[i:i+BATCH_SIZE]
    outputs = keyword_extract.batch(batch)
    for inp, out in zip(batch, outputs):
        results_100.append({"review": inp["review"], "text": out})

Extracting keywords: 100%|██████████| 25/25 [03:51<00:00,  9.25s/it]


In [ ]:
for result in results_100[0:10]:
    print(result['text']['keywords'])

ช็อกโกแลต, เค้ก, ขนมหวาน
สนามบิน, ร้านอาหาร, บรรยากาศ
กาแฟ, เค้ก, บรรยากาศ
อาหารอิตาเลียน, บรรยากาศ, พิซซ่า
บรรยากาศ, ฟิวชั่น, ราคา
บรรยากาศ, ส้มตำ, รสชาติ
กาแฟ, เอ็กซ์เพรสโซ่, กลิ่น
วาฟเฟิล, กาแฟ, ไซรัป
กาแฟ, โปรโมชั่น, คนเยอะ
อาหารฝรั่ง, ชีสทอด, ฮันนี่โทส


In [ ]:
context_prompt = PromptTemplate(
    template="""จาก Review ต่อไปนี้
    Extract context ใน Review ที่เป็นตัวแทนที่บรรยายคำว่า {keyword} ได้ดีที่สุด

    Review: {review}

    ตอบผลลัพธ์ในรูปแบบ JSON โดยมีคีย์เป็น 'context'
    และค่าเป็นข้อความที่ดึงได้จาก Review

    ตัวอย่าง
    keyword คือ "กาแฟ"
    context: "กาแฟที่นี่อร่อยมาก หอมกลิ่นกาแฟคั่วสด"

    หากไม่พบข้อมูลให้ตอบ 'ไม่ระบุ'
    """
)

In [ ]:
sentiment_prompt = PromptTemplate(
    template="""จากข้อความต่อไปนี้ ให้วิเคราะห์ว่าเป็น sentiment แบบไหน
    เลือกจากตัวเลือกเหล่านี้เท่านั้น: "positive", "negative", "neutral"

    ข้อความ: {message}

    ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'sentiment'
    และค่าเป็นหนึ่งใน list ต่อไปนี้ [positive, negative, neutral]

    คำแนะนำ
    - positive คือ ข้อความที่แสดงความพอใจ ชอบ ดี
    - negative คือ ข้อความที่แสดงความไม่พอใจ ไม่ชอบ แย่
    - neutral คือ ข้อความที่ไม่แสดงความรู้สึกชัดเจน เป็นกลาง

    # ตัวอย่าง
    # ข้อความ "อร่อยมาก" → {{"sentiment": "positive"}}
    # ข้อความ "แพงไป" → {{"sentiment": "negative"}}
    # ข้อความ "ปกติ" → {{"sentiment": "neutral"}}
    """
)


In [ ]:
class ContextKeyword(BaseModel):
    context: str = Field(..., description="context that best represent this keyword")

class ContexExtraction(BaseModel):
    extracted: ContextKeyword

context_parser = JsonOutputParser(pydantic_object=ContexExtraction)

In [ ]:
context_extract = context_prompt | llm3 | context_parser

In [ ]:
class Sentiment(BaseModel):
    sentiment: str = Field(..., description="sentiment of review")

class SentimentExtraction(BaseModel):
    extracted: Sentiment


sentiment_parser = JsonOutputParser(pydantic_object=SentimentExtraction)

In [ ]:
sentiment_extract = sentiment_prompt | llm3 | sentiment_parser

In [ ]:
results_100[0]['text']['keywords'].split(',')

['ช็อกโกแลต', ' เค้ก', ' ขนมหวาน']

In [ ]:
context_inputs = []

for review_data in tqdm(results_100, desc="Preparing context inputs"):
    review = review_data['review']
    keywords = review_data['text']['keywords'].split(',')
    for keyword in keywords:
        context_inputs.append({"review": review, "keyword": keyword.strip()})

Preparing context inputs: 100%|██████████| 400/400 [00:00<00:00, 255283.26it/s]


In [ ]:
len(context_inputs)

1200

In [ ]:
context_results = []
for i in tqdm(range(len(results_100)), desc="Extracting contexts"):
   result = context_extract.batch(context_inputs[i*3:(i*3)+3])
   contexts_list = [d['context'] for d in result]
   results_100[i]['contexts'] = contexts_list

Extracting contexts: 100%|██████████| 400/400 [27:13<00:00,  4.08s/it]


In [ ]:
results_100[:2]

[{'review': 'Chocolate crape cake with strawberry 145฿\nDark chocolate Ganache cake 145฿\n\nบริการแบบ งงๆ แบบมีหลายเคาร์เตอร์ \n\nขนมอร่อย ร้านตกแต่งสวย มีทั้งอาหาร ขนม กาแฟ มีของ grocery ให้ซื้อไป cook เองที่บ้านได้ \n\nอิ่มคาวมาจากร้านข้างบนแล้ว มาต่อแค่ขนมหวาน 2 ชิ้นเท่านั้น แต่ฟินง่ะ\n\nแอบอ่านรีวิวก่อนหน้า ส่วนใหญ่จะ complain อาหารคาว',
  'text': {'keywords': 'ช็อกโกแลต, เค้ก, ขนมหวาน'},
  'contexts': ['ช็อกโกแลต',
   'เค้กช็อกโกแลตครีมเค้กช็อกโกแลตมูสเค้ก',
   'ขนมอร่อย']},
 {'review': 'ถ้ามีโอกาสมาสนามบิน ห้ามพลาด ร้านอาหารเล็กๆ มีให้เลือกทั้งก๋วยเตี๋ยว ส้มตำ น้ำตก อาหารจานเดียว หรือกาแฟสด น้ำผลไม้ต่างๆ ให้เลือกหลากหลาย อีกทั้งบรรยากาศดีเหมือนอยู่ในท้องนา ภายในสนามบินยังมีสินค้าจำหน่ายย่อมๆ ให้ซื้อเป็นที่ระลึก และยังมีสวนสัตว์ย่อมๆ ไว้ให้เพลิดเพลินด้วย',
  'text': {'keywords': 'สนามบิน, ร้านอาหาร, บรรยากาศ'},
  'contexts': ['สนามบิน', 'ร้านอาหารเล็กๆ', 'บรรยากาศดีเหมือนอยู่ในท้องนา']}]

In [ ]:
sentiment_inputs = []

for review_data in tqdm(results_100, desc="Preparing sentiment inputs"):
    contexts = review_data['contexts']
    for context in contexts:
        sentiment_inputs.append({"message": context})

Preparing sentiment inputs: 100%|██████████| 400/400 [00:00<00:00, 206565.08it/s]


In [ ]:
len(sentiment_inputs)

1200

In [ ]:
sentiment_inputs[:6]

[{'message': 'ช็อกโกแลต'},
 {'message': 'เค้กช็อกโกแลตครีมเค้กช็อกโกแลตมูสเค้ก'},
 {'message': 'ขนมอร่อย'},
 {'message': 'สนามบิน'},
 {'message': 'ร้านอาหารเล็กๆ'},
 {'message': 'บรรยากาศดีเหมือนอยู่ในท้องนา'}]

In [ ]:
for i in tqdm(range(len(results_100)), desc="Extracting sentiment"):
   result = sentiment_extract.batch(sentiment_inputs[i*3:(i*3)+3])
   sentiment_list = [d['sentiment'] for d in result]
   results_100[i]['sentiment'] = sentiment_list

Extracting sentiment: 100%|██████████| 400/400 [17:24<00:00,  2.61s/it]


In [ ]:
results_100[0:2]

[{'review': 'Chocolate crape cake with strawberry 145฿\nDark chocolate Ganache cake 145฿\n\nบริการแบบ งงๆ แบบมีหลายเคาร์เตอร์ \n\nขนมอร่อย ร้านตกแต่งสวย มีทั้งอาหาร ขนม กาแฟ มีของ grocery ให้ซื้อไป cook เองที่บ้านได้ \n\nอิ่มคาวมาจากร้านข้างบนแล้ว มาต่อแค่ขนมหวาน 2 ชิ้นเท่านั้น แต่ฟินง่ะ\n\nแอบอ่านรีวิวก่อนหน้า ส่วนใหญ่จะ complain อาหารคาว',
  'text': {'keywords': 'ช็อกโกแลต, เค้ก, ขนมหวาน'},
  'contexts': ['ช็อกโกแลต',
   'เค้กช็อกโกแลตครีมเค้กช็อกโกแลตมูสเค้ก',
   'ขนมอร่อย'],
  'sentiment': ['neutral', 'neutral', 'positive']},
 {'review': 'ถ้ามีโอกาสมาสนามบิน ห้ามพลาด ร้านอาหารเล็กๆ มีให้เลือกทั้งก๋วยเตี๋ยว ส้มตำ น้ำตก อาหารจานเดียว หรือกาแฟสด น้ำผลไม้ต่างๆ ให้เลือกหลากหลาย อีกทั้งบรรยากาศดีเหมือนอยู่ในท้องนา ภายในสนามบินยังมีสินค้าจำหน่ายย่อมๆ ให้ซื้อเป็นที่ระลึก และยังมีสวนสัตว์ย่อมๆ ไว้ให้เพลิดเพลินด้วย',
  'text': {'keywords': 'สนามบิน, ร้านอาหาร, บรรยากาศ'},
  'contexts': ['สนามบิน', 'ร้านอาหารเล็กๆ', 'บรรยากาศดีเหมือนอยู่ในท้องนา'],
  'sentiment': ['neutral', 'neutral', 'posit

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        'review': r['review'],
        **{f'keyword{i+1}': k.strip() for i, k in enumerate(r['text']['keywords'])},
        **{f'context{i+1}_keyword': c for i, c in enumerate(r['contexts'])},
        **{f'sentiment{i+1}': s for i, s in enumerate(r['sentiment'])}
    }
    for r in results_100
])

In [ ]:
df.to_csv('results.csv', index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd

# รวมไฟล์ทั้งหมดเข้าด้วยกัน
files = [
    "results_part1.csv",
    "results_part2.csv",
    "results_part3.csv",
    "results_part4.csv",
    "results_part5.csv"
]

# อ่านและต่อไฟล์เข้าด้วยกัน
combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# บันทึกเป็นไฟล์เดียว
combined.to_csv("results_all_1000.csv", index=False, encoding="utf-8-sig")

print("results_all_1000.csv")


results_all_1000.csv
